In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

# --------------------------------------------------
# STEP 1: LOAD DATA
# --------------------------------------------------

df = pd.read_csv("C:/Users/Hp/Downloads/customer_churn_dataset-testing-master.csv")
df.columns = df.columns.str.strip()

In [2]:
print(df.columns)

Index(['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency',
       'Support Calls', 'Payment Delay', 'Subscription Type',
       'Contract Length', 'Total Spend', 'Last Interaction', 'Churn'],
      dtype='str')


In [3]:
target = "Churn"

numeric_features = [
    "Age",
    "Tenure",
    "Usage Frequency",
    "Support Calls",
    "Payment Delay",
    "Total Spend",
    "Last Interaction"
]

categorical_features = [
    "Gender",
    "Subscription Type",
    "Contract Length"
]

# Exclude rows with missing target values
df = df.dropna(subset=[target]).copy()

print("Dataset shape:", df.shape)
print("\nChurn distribution:")
print(df[target].value_counts())
print("\nChurn rate (%):", round(df[target].mean() * 100, 2))

Dataset shape: (64374, 12)

Churn distribution:
Churn
0    33881
1    30493
Name: count, dtype: int64

Churn rate (%): 47.37


In [6]:
# --------------------------------------------------
# STEP 2: DEFINE FEATURES AND TARGET
# --------------------------------------------------

X = df[numeric_features + categorical_features].copy()
y = df[target].copy()

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget variable: Churn")
print("Features:", X.columns.tolist())


Feature matrix shape: (64374, 10)
Target shape: (64374,)

Target variable: Churn
Features: ['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction', 'Gender', 'Subscription Type', 'Contract Length']


In [7]:
# --------------------------------------------------
# STEP 3: WELCH'S INDEPENDENT SAMPLES T-TEST
# Compare numeric features between churn groups
# --------------------------------------------------

ttest_results = []

for col in numeric_features:
    retained = df.loc[df[target] == 0, col].dropna()
    churned = df.loc[df[target] == 1, col].dropna()

    # Welch's t-test does not require equal variances
    t_stat, p_value = stats.ttest_ind(
        churned,
        retained,
        equal_var=False
    )

    ttest_results.append({
        "Feature": col,
        "Retained Mean": retained.mean(),
        "Churned Mean": churned.mean(),
        "Mean Difference (Churned - Retained)": (
            churned.mean() - retained.mean()
        ),
        "T Statistic": t_stat,
        "P Value": p_value
    })

ttest_df = pd.DataFrame(ttest_results)

# Correct for testing multiple numerical features
ttest_df["Adjusted P Value"] = multipletests(
    ttest_df["P Value"],
    method="fdr_bh"
)[1]

ttest_df["Significant"] = np.where(
    ttest_df["Adjusted P Value"] < 0.05,
    "Yes",
    "No"
)

print("\n========== WELCH'S T-TEST RESULTS ==========")
print(ttest_df.round(4).to_string(index=False))


========== WELCH'S T-TEST RESULTS ==========
         Feature  Retained Mean  Churned Mean  Mean Difference (Churned - Retained)  T Statistic  P Value  Adjusted P Value Significant
             Age        41.1327       42.9024                                1.7697      16.1138   0.0000            0.0000         Yes
          Tenure        28.8265       35.5152                                6.6887      50.7628   0.0000            0.0000         Yes
 Usage Frequency        16.0429       14.0106                               -2.0323     -29.2695   0.0000            0.0000         Yes
   Support Calls         4.5008        6.4006                                1.8999      81.7878   0.0000            0.0000         Yes
   Payment Delay        12.4531       22.3349                                9.8818     173.0250   0.0000            0.0000         Yes
     Total Spend       560.5420      519.3361                              -41.2058     -20.0687   0.0000            0.0000         Yes
La

In [8]:
# --------------------------------------------------
# STEP 4: CHI-SQUARE TEST OF INDEPENDENCE
# Compare categorical features with churn
# --------------------------------------------------

chi_results = []

for col in categorical_features:
    temp = df[[col, target]].dropna()

    contingency_table = pd.crosstab(
        temp[col],
        temp[target]
    )

    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )

    # Cramer's V effect size
    n = contingency_table.to_numpy().sum()
    k = min(contingency_table.shape)

    cramers_v = (
        np.sqrt(chi2 / (n * (k - 1)))
        if k > 1 else np.nan
    )

    chi_results.append({
        "Feature": col,
        "Chi-Square": chi2,
        "Degrees of Freedom": dof,
        "P Value": p_value,
        "Cramer's V": cramers_v,
        "Minimum Expected Count": expected.min(),
        "Significant": "Yes" if p_value < 0.05 else "No"
    })

    print(f"\nContingency table for {col}:")
    print(contingency_table)

chi_df = pd.DataFrame(chi_results)

chi_df["Adjusted P Value"] = multipletests(
    chi_df["P Value"],
    method="fdr_bh"
)[1]

chi_df["Significant After Correction"] = np.where(
    chi_df["Adjusted P Value"] < 0.05,
    "Yes",
    "No"
)

print("\n========== CHI-SQUARE RESULTS ==========")
print(chi_df.round(4).to_string(index=False))


Contingency table for Gender:
Churn       0      1
Gender              
Female  15442  18911
Male    18439  11582

Contingency table for Subscription Type:
Churn                  0      1
Subscription Type              
Basic              11095  10356
Premium            11461   9960
Standard           11325  10177

Contingency table for Contract Length:
Churn                0      1
Contract Length              
Annual           11515   9895
Monthly          10709  11421
Quarterly        11657   9177

========== CHI-SQUARE RESULTS ==========
          Feature  Chi-Square  Degrees of Freedom  P Value  Cramer's V  Minimum Expected Count Significant  Adjusted P Value Significant After Correction
           Gender   1742.3455                   1   0.0000      0.1645              14220.4982         Yes            0.0000                          Yes
Subscription Type     13.6559                   2   0.0011      0.0146              10146.8070         Yes            0.0011                   

In [10]:
# --------------------------------------------------
# STEP 5: ONE-WAY ANOVA
# Example: Does average tenure differ by subscription type?
# --------------------------------------------------

anova_feature = "Tenure"
group_feature = "Subscription Type"

anova_data = df[[anova_feature, group_feature]].dropna()

groups = [
    group[anova_feature].values
    for _, group in anova_data.groupby(group_feature)
]

# Descriptive statistics by group
print("\n========== ANOVA GROUP SUMMARY ==========")
print(
    anova_data.groupby(group_feature)[anova_feature]
    .agg(["count", "mean", "std"])
    .round(3)
)

# Check homogeneity of variances
levene_stat, levene_p = stats.levene(*groups)

# Run standard one-way ANOVA
f_stat, anova_p = stats.f_oneway(*groups)

print("\n========== ONE-WAY ANOVA RESULTS ==========")
print("Feature tested:", anova_feature)
print("Grouping variable:", group_feature)
print("Levene statistic:", round(levene_stat, 4))
print("Levene p-value:", round(levene_p, 6))
print("ANOVA F-statistic:", round(f_stat, 4))
print("ANOVA p-value:", round(anova_p, 6))

if anova_p < 0.05:
    print("Conclusion: At least one group mean differs significantly.")
else:
    print("Conclusion: Insufficient evidence that group means differ.")

if levene_p < 0.05:
    print(
        "Warning: Group variances differ. Consider Welch's ANOVA "
        "instead of standard ANOVA."
    )


========== ANOVA GROUP SUMMARY ==========
                   count    mean     std
Subscription Type                       
Basic              21451  31.942  17.162
Premium            21421  31.925  17.066
Standard           21502  32.117  17.067

========== ONE-WAY ANOVA RESULTS ==========
Feature tested: Tenure
Grouping variable: Subscription Type
Levene statistic: 0.5195
Levene p-value: 0.594849
ANOVA F-statistic: 0.835
ANOVA p-value: 0.433871
Conclusion: Insufficient evidence that group means differ.


In [11]:
# --------------------------------------------------
# STEP 6: RANK FEATURES BY STATISTICAL EVIDENCE
# --------------------------------------------------

print("\n========== NUMERICAL FEATURES ASSOCIATED WITH CHURN ==========")

print(
    ttest_df.sort_values("Adjusted P Value")[
        [
            "Feature",
            "Retained Mean",
            "Churned Mean",
            "Adjusted P Value",
            "Significant"
        ]
    ].round(4).to_string(index=False)
)

print("\n========== CATEGORICAL FEATURES ASSOCIATED WITH CHURN ==========")

print(
    chi_df.sort_values("Adjusted P Value")[
        [
            "Feature",
            "Cramer's V",
            "Adjusted P Value",
            "Significant After Correction"
        ]
    ].round(4).to_string(index=False)
)



========== NUMERICAL FEATURES ASSOCIATED WITH CHURN ==========
         Feature  Retained Mean  Churned Mean  Adjusted P Value Significant
          Tenure        28.8265       35.5152            0.0000         Yes
   Support Calls         4.5008        6.4006            0.0000         Yes
   Payment Delay        12.4531       22.3349            0.0000         Yes
 Usage Frequency        16.0429       14.0106            0.0000         Yes
     Total Spend       560.5420      519.3361            0.0000         Yes
             Age        41.1327       42.9024            0.0000         Yes
Last Interaction        15.5219       15.4732            0.4747          No

========== CATEGORICAL FEATURES ASSOCIATED WITH CHURN ==========
          Feature  Cramer's V  Adjusted P Value Significant After Correction
           Gender      0.1645            0.0000                          Yes
  Contract Length      0.0639            0.0000                          Yes
Subscription Type      0.0146  